# Lesson 13A: Diffusion Models - Theory

Learn state-of-the-art generative models.

**Diffusion Models** generate data by reversing a noising process.

**Forward Process (noising):**
- Gradually add Gaussian noise over T timesteps
- q(x_t | x_{t-1}) = N(x_t; √(1-β_t) x_{t-1}, β_t I)

**Reverse Process (denoising):**
- Learn to remove noise step-by-step
- p_θ(x_{t-1} | x_t) = N(x_{t-1}; μ_θ(x_t, t), Σ_θ(x_t, t))

**Training:** Predict noise ε added at each timestep

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("✅ Loaded!")

## DDPM Algorithm (Denoising Diffusion Probabilistic Models)

**Forward (fixed):**
```
x_t = √(α_t) x_0 + √(1 - α_t) ε, where ε ~ N(0, I)
```

**Training objective:**
```
L = E_{t,x_0,ε} [ ||ε - ε_θ(x_t, t)||² ]
```

**Sampling (reverse):**
1. Start with x_T ~ N(0, I)
2. For t = T to 1:
   - Predict noise ε_θ(x_t, t)
   - Denoise: x_{t-1} = (x_t - √(1-α_t) ε_θ) / √α_t + σ_t z
3. Return x_0

**Key Insight:** Neural network learns to predict and remove noise.

In [ ]:
import torch
import torch.nn as nn

class SimpleDiffusionModel(nn.Module):
    """Simplified noise prediction network"""
    def __init__(self, input_dim=2, time_dim=32, hidden_dim=64):
        super().__init__()
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(input_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def forward(self, x, t):
        # t is timestep, embed it
        t_embed = self.time_embed(t.unsqueeze(-1))
        # Concatenate x and time embedding
        x_t = torch.cat([x, t_embed], dim=-1)
        # Predict noise
        return self.net(x_t)

# Noise schedule (linear)
def get_beta_schedule(T=100):
    return torch.linspace(0.0001, 0.02, T)

def forward_diffusion(x0, t, betas):
    """Add noise to x0 at timestep t"""
    alpha = 1 - betas
    alpha_bar = torch.cumprod(alpha, dim=0)
    
    alpha_t = alpha_bar[t]
    noise = torch.randn_like(x0)
    
    x_t = torch.sqrt(alpha_t) * x0 + torch.sqrt(1 - alpha_t) * noise
    return x_t, noise

# Test forward diffusion
x0 = torch.randn(10, 2)
betas = get_beta_schedule(T=100)
t = torch.tensor([50])

x_t, noise = forward_diffusion(x0, t, betas)

print(f"Original shape: {x0.shape}")
print(f"Noised shape: {x_t.shape}")
print(f"Noise shape: {noise.shape}")
print("✅ Forward diffusion process defined!")

# Visualize noising process
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
timesteps = [0, 25, 50, 75, 99]
x_test = torch.randn(500, 2)
for i, t in enumerate(timesteps):
    t_tensor = torch.tensor([t])
    x_noised, _ = forward_diffusion(x_test, t_tensor, betas)
    axes[i].scatter(x_noised[:, 0], x_noised[:, 1], s=5, alpha=0.5)
    axes[i].set_title(f't = {t}')
    axes[i].set_xlim(-3, 3)
    axes[i].set_ylim(-3, 3)
plt.suptitle('Forward Diffusion Process', fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ Visualized progressive noising!")